# Concatenate Datasets

In [1]:
import anndata as ad
import pandas as pd

# Load CellTypist Dataset
adata_celltypist = ad.io.read_h5ad('../data/CellTypistDataset/CountAdded_PIP_global_object_for_cellxgene_annotated_fine_grained.h5ad')

# Filter blood data
adata_celltypist = adata_celltypist[adata_celltypist.obs['Organ'] == 'BLD'].copy()

# Use raw data instead of already preprocessed data
adata_celltypist.X = adata_celltypist.layers['counts'].copy()

print(adata_celltypist)

AnnData object with n_obs × n_vars = 27620 × 36473
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype', 'Sex', 'Age_range', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'Age_range_colors', 'Sex_colors', 'scrublet'
    obsm: 'X_umap'
    layers: 'counts'


In [2]:
# Load Human Cell Atlas
adata_hca = ad.io.read_h5ad('../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated_fine_grained_more_donors.h5ad')

# Use raw data instead of already preprocessed data
adata_hca = adata_hca.raw.to_adata()

print(adata_hca)

AnnData object with n_obs × n_vars = 39996 × 25825
    obs: 'n_genes', 'Channel', 'n_counts', 'percent_mito', 'scale', 'Group', 'leiden_labels', 'Donor', 'doublet_score', 'pred_dbl', 'anno', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'predicted_doublet', 'scumi-annotation'
    var: 'featureid'
    uns: 'Channels', 'Groups', 'PCs', 'W_pca_harmony', 'c2gid', 'df_qcplot', 'genome', 'gncells', 'leiden_resolution', 'modality', 'ncells', 'norm_count', 'pca', 'pca_features', 'pca_harmony_knn_distances', 'pca_harmony_knn_indices', 'scrublet', 'stdzn_max_value', 'stdzn_mean', 'stdzn_std'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'


In [3]:
# Check common genes
common_genes = adata_celltypist.var_names.intersection(adata_hca.var_names)
print(f"Common Genes: {len(common_genes)}")

Common Genes: 24668


In [4]:
# Merge Datasets
adata_hca.varm = {}
adata_celltypist.varm = {}

adata_merged = ad.concat(
    {"CellTypist": adata_celltypist, "HCA": adata_hca},
    axis=0,
    join="inner",
    label="dataset",
    merge="unique"
)

adata_merged.obs_names_make_unique()
print(adata_merged)

AnnData object with n_obs × n_vars = 67616 × 24668
    obs: 'Donor', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation', 'dataset'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'featureid'
    obsm: 'X_umap'


In [6]:
# Remove Unassigned Class
adata_merged = adata_merged[adata_merged.obs['scumi-annotation'] != 'Unassigned'].copy()
adata_merged.obs['scumi-annotation'] = adata_merged.obs['scumi-annotation'].cat.remove_unused_categories()

In [7]:
ad.settings.allow_write_nullable_strings = True

# Save merged dataset
adata_merged.write(filename='../data/CellTypist_HumanCellAtlas_Merged.h5ad')

In [8]:
import anndata as ad
import pandas as pd

adata = ad.io.read_h5ad('../data/CellTypist_HumanCellAtlas_Merged.h5ad')

print(adata)


classes = adata.obs['scumi-annotation']

classes_series = pd.Series(classes)

class_counts = classes_series.value_counts()
print("Number of classes present: ", len(class_counts))
print(class_counts)

AnnData object with n_obs × n_vars = 67274 × 24668
    obs: 'Donor', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'scumi-annotation', 'dataset'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'featureid'
    obsm: 'X_umap'
Number of classes present:  26
scumi-annotation
CD4 Naive T cell               9580
CD14+ Monocyte                 9170
CD8 TCM                        6325
CD8 TEM                        5933
Naive B cell                   4854
CD56-dim NK cell               4692
CD8 Naive 